# WaveCoAtNet Account 3: 5-Fold Cross-Validation
> Run this in Account 3. Set Runtime > GPU before starting.
> All outputs auto-save to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q roboflow timm torchinfo seaborn tqdm scikit-learn fvcore

import os
WORK = '/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet/Wave-CoAtNet-Ichthyosis/wave-CoAtNet'
REPO_ROOT = '/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet'

if not os.path.exists(REPO_ROOT):
    os.makedirs('/content/drive/MyDrive/WaveCoAtNet_experiments', exist_ok=True)
    %cd /content/drive/MyDrive/WaveCoAtNet_experiments
    !git clone https://github.com/Cyrax321/wave-coAtNet.git
else:
    %cd {REPO_ROOT}
    !git pull origin main

%cd {WORK}


In [ ]:
import os, csv, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, confusion_matrix)
from sklearn.preprocessing import label_binarize
from IPython.display import display, Image

os.chdir('/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet/Wave-CoAtNet-Ichthyosis/wave-CoAtNet')
os.makedirs('figures', exist_ok=True)

class_names = ['Harlequin ichthyosis', 'Healthy skin', 'Ichthyosis vulgaris',
               'Lamellar ichthyosis', 'Netherton syndrome']
n_classes = len(class_names)

# Summary
print("="*60)
print("  CROSS-VALIDATION SUMMARY")
print("="*60)
if os.path.exists('crossval_summary.txt'):
    with open('crossval_summary.txt') as f: print(f.read())

if os.path.exists('crossval_results.csv'):
    with open('crossval_results.csv') as f: print(f.read())

if os.path.exists('mcnemar_results.csv'):
    print("\n--- McNemar Results ---")
    with open('mcnemar_results.csv') as f: print(f.read())

# Headline claim with CI
if os.path.exists('crossval_results.csv'):
    with open('crossval_results.csv') as f:
        rows = list(csv.DictReader(f))
    accs = [float(r['accuracy']) for r in rows]
    f1s = [float(r['macro_f1']) for r in rows]
    ci95 = 1.96 * np.std(accs, ddof=1) / np.sqrt(len(accs))
    print(f"\n--- HEADLINE CLAIM ---")
    print(f"  Accuracy: {np.mean(accs)*100:.2f}% +/- {np.std(accs, ddof=1)*100:.2f}%")
    print(f"  95% CI:   [{(np.mean(accs)-ci95)*100:.2f}%, {(np.mean(accs)+ci95)*100:.2f}%]")
    print(f"  Macro F1: {np.mean(f1s):.4f} +/- {np.std(f1s, ddof=1):.4f}")

# Cohen's Kappa + Sensitivity/Specificity from cross-val folds
all_yt, all_yp = [], []
for k in range(1, 6):
    if os.path.exists(f'fold_{k}_y_true.npy'):
        all_yt.append(np.load(f'fold_{k}_y_true.npy'))
        all_yp.append(np.load(f'fold_{k}_y_pred.npy'))
if all_yt:
    yt = np.concatenate(all_yt)
    yp = np.concatenate(all_yp)
    print(f"\nCohen's Kappa: {cohen_kappa_score(yt, yp):.4f}")

    cm = confusion_matrix(yt, yp, labels=range(n_classes))
    print(f"\n{'Class':<25s} {'Sensitivity':>12s} {'Specificity':>12s}")
    print("-"*50)
    for i, cls in enumerate(class_names):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"  {cls:<23s} {sens:>11.4f} {spec:>11.4f}")

# Cross-val bar chart
folds = [int(r['fold']) for r in rows]
accs_pct = [float(r['accuracy'])*100 for r in rows]
f1s_pct = [float(r['macro_f1'])*100 for r in rows]
wf1s_pct = [float(r['weighted_f1'])*100 for r in rows]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(folds))
w = 0.3
bars1 = ax.bar(x - w, accs_pct, w, label='Accuracy (%)', color='#2563EB', alpha=0.85)
bars2 = ax.bar(x, f1s_pct, w, label='Macro F1 (%)', color='#DC2626', alpha=0.85)
bars3 = ax.bar(x + w, wf1s_pct, w, label='Weighted F1 (%)', color='#16A34A', alpha=0.85)
ax.axhline(np.mean(accs_pct), color='#2563EB', linestyle='--', alpha=0.5, label=f'Mean Acc ({np.mean(accs_pct):.1f}%)')
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {f}' for f in folds], fontsize=10)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('5-Fold Stratified Cross-Validation: WaveCoAtNet', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(min(min(accs_pct), min(f1s_pct)) - 5, 105)
plt.tight_layout()
plt.savefig('figures/crossval_folds_bar.png', dpi=300)
plt.close()
print("\nSaved: figures/crossval_folds_bar.png")

# Box plot
fig, ax = plt.subplots(figsize=(8, 6))
data = [accs_pct, f1s_pct, wf1s_pct]
bp = ax.boxplot(data, patch_artist=True, labels=['Accuracy', 'Macro F1', 'Weighted F1'],
                widths=0.5, showmeans=True, meanprops=dict(marker='D', markerfacecolor='white', markersize=8))
for patch, color in zip(bp['boxes'], ['#2563EB', '#DC2626', '#16A34A']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for i, d in enumerate(data):
    jitter = np.random.normal(0, 0.04, len(d))
    ax.scatter([i+1+j for j in jitter], d, color='black', alpha=0.6, s=30, zorder=5)
    ax.text(i+1, max(d)+1.5, f'{np.mean(d):.1f}+/-{np.std(d, ddof=1):.1f}%',
            ha='center', fontsize=8, fontweight='bold')
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Cross-Validation Score Distribution', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/crossval_boxplot.png', dpi=300)
plt.close()
print("Saved: figures/crossval_boxplot.png")

# Fold CM grid
fold_cms = [f'fold_{k}_cm.png' for k in range(1, 6) if os.path.exists(f'fold_{k}_cm.png')]
if fold_cms:
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    for i, cm_f in enumerate(fold_cms):
        axes[i].imshow(plt.imread(cm_f)); axes[i].axis('off')
        axes[i].set_title(f'Fold {i+1}', fontsize=11, fontweight='bold')
    fig.suptitle('Per-Fold Confusion Matrices', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/crossval_cm_grid.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: figures/crossval_cm_grid.png")

for f in ['figures/crossval_folds_bar.png', 'figures/crossval_boxplot.png', 'figures/crossval_cm_grid.png']:
    if os.path.exists(f):
        display(Image(filename=f, width=700))

print("\n--- Saved Files ---")
for f in sorted(glob.glob('crossval_*') + glob.glob('mcnemar_*') + glob.glob('fold_*')):
    print(f"  {f} ({os.path.getsize(f)/1024:.1f} KB)")

In [ ]:
import os
os.chdir('/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet/Wave-CoAtNet-Ichthyosis/wave-CoAtNet')

print("="*60)
print("  PUBLICATION READINESS CHECK")
print("="*60)

preds = sorted([f for f in os.listdir('.') if f.endswith('_y_pred.npy') and not f.startswith('ablation')])
print(f"\n[1] Model predictions: {len(preds)}/12")
for p in preds: print(f"    [OK] {p}")
missing_models = {'wavecoatnet','coatnet','efficientnet_pretrained','efficientnet_scratch',
                  'swin_pretrained','swin_scratch','vit_pretrained','vit_scratch',
                  'gft','biomedclip','dinov2','cnn'}
for m in sorted(missing_models - {p.replace('_y_pred.npy','') for p in preds}):
    print(f"    [MISSING] {m}_y_pred.npy")

abl = sorted([f for f in os.listdir('.') if f.startswith('ablation_') and f.endswith('_y_pred.npy')])
print(f"\n[2] Ablation predictions: {len(abl)}/10")
for p in abl: print(f"    [OK] {p}")

print(f"\n[3] Data tables:")
for csv_f in ['ablation_results.csv','crossval_summary.txt','crossval_results.csv',
              'mcnemar_results.csv','ablation_mcnemar.csv','figures/full_comparison_table.csv',
              'figures/inference_timing.csv','figures/mcnemar_pvalues.csv','figures/comprehensive_results.csv']:
    status = 'OK' if os.path.exists(csv_f) else 'MISSING'
    print(f"    [{status}] {csv_f}")

fig_count = 0
print(f"\n[4] Publication figures:")
if os.path.exists('figures'):
    for f in sorted(os.listdir('figures')):
        print(f"    [OK] figures/{f}"); fig_count += 1

gc_count = 0
print(f"\n[5] Grad-CAM:")
if os.path.exists('gradcam'):
    for f in sorted(os.listdir('gradcam')):
        print(f"    [OK] gradcam/{f}"); gc_count += 1

total_preds = len(preds) + len(abl)
if total_preds >= 22 and fig_count >= 20 and gc_count > 0:
    print(f"\n{'='*60}")
    print("  ALL COMPLETE -- READY TO WRITE THE PAPER")
    print(f"{'='*60}")
else:
    print(f"\n{'='*60}")
    print(f"  INCOMPLETE: preds={total_preds}/22  figs={fig_count}/20+  gradcam={'OK' if gc_count else 'MISSING'}")
    print(f"{'='*60}")